# Week 01 · Python 工程化 / Engineering Python

虚拟环境让一个项目的依赖版本不会污染另一个项目。Windows 用 `python -m venv .venv` 创建，`.venv/Scripts/python -m pip install ...` 安装；不要把虚拟环境提交到 Git。模块负责可复用逻辑，入口负责解析参数，数据目录保存运行时文件。类型注解帮助读者和静态检查工具，但不会自动阻止错误输入，所以边界仍要校验。

下面使用 dataclass 表达领域对象、pathlib 处理路径、JSON 保存数据、CSV 导出表格、argparse 解析命令。写入时先写临时文件再替换，避免半份 JSON；这只能提高单进程文件可靠性，不能替代多用户数据库事务。异常在能给出明确处理的地方捕获，损坏文件不应直接被空列表覆盖。

Git 工作区 → 暂存区 → 提交：`git diff` 检查修改，`git add 文件` 选择提交内容，`git commit -m "feat: persist tasks"` 记录快照。`git switch -c feature/todo` 建分支；合并冲突要读两边业务含义再修改，不是盲选一方。练习五次真实提交，不用脚本伪造历史。环境变量适合注入配置；密钥不写进源码或日志。

## 学习方式 / How to study
先预测代码结果，再逐行运行。改变一个输入、解释变化，最后不看参考实现重写关键函数。阅读不是掌握的证据；能独立实现、测试、解释失败才是。

In [ ]:
# 所有输出放在本 Notebook 内核工作目录，重复运行不会破坏其他周数据。
from dataclasses import dataclass, asdict
from pathlib import Path
import argparse, csv, json, os

@dataclass
class Task:
    id: int
    title: str
    done: bool = False

class JsonTasks:
    def __init__(self, path: Path):
        self.path = path

    def load(self) -> list[Task]:
        if not self.path.exists():
            return []
        # 不吞掉 JSONDecodeError，否则损坏的数据可能被静默覆盖。
        rows = json.loads(self.path.read_text(encoding="utf-8"))
        return [Task(**row) for row in rows]

    def save(self, tasks: list[Task]) -> None:
        self.path.parent.mkdir(parents=True, exist_ok=True)
        temporary = self.path.with_suffix(".tmp")
        temporary.write_text(json.dumps([asdict(t) for t in tasks],
                                       ensure_ascii=False, indent=2), encoding="utf-8")
        temporary.replace(self.path)

    def add(self, title: str) -> Task:
        title = title.strip()
        if not title:
            raise ValueError("标题不能为空")
        tasks = self.load()
        task = Task(max((t.id for t in tasks), default=0)+1, title)
        self.save(tasks+[task])
        return task

    def complete(self, identifier: int) -> None:
        tasks = self.load()
        target = next((t for t in tasks if t.id == identifier), None)
        if target is None:
            raise KeyError(identifier)
        target.done = True
        self.save(tasks)

# 用单独的演示文件确保每次执行可复现；真实 CLI 不应在启动时清空数据库。
repository = JsonTasks(Path("week01-demo/tasks.json"))
repository.save([])
parser = argparse.ArgumentParser()
parser.add_argument("title")
arguments = parser.parse_args(["学习 Git 与 JSON"])
created = repository.add(arguments.title)
repository.complete(created.id)
assert repository.load()[0].done
try:
    repository.add("  ")
except ValueError as error:
    print("预期的输入错误：", error)
with Path("week01-demo/tasks.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["id", "title", "done"])
    writer.writeheader()
    writer.writerows(asdict(t) for t in repository.load())
print("Python 环境：", os.environ.get("VIRTUAL_ENV", "由工作台显式选择解释器"))
print(repository.load())

## 练习 / Exercises
实现不区分大小写的搜索；保留 load 的损坏文件异常。增加删除功能并验证删除不存在 ID 的行为。

先在下面独立完成，再展开参考实现。

In [ ]:
# 在这里写你的实现；运行后检查边界。


## 参考实现与验收 / Reference and checks
参考实现是一个可行方案，不是唯一答案。不要在未完成练习前直接复制。

In [ ]:
def search_tasks(repository, keyword):
    return [task for task in repository.load() if keyword.casefold() in task.title.casefold()]
assert len(search_tasks(repository, "git")) == 1
def delete_task(repository, identifier):
    tasks = repository.load()
    filtered = [task for task in tasks if task.id != identifier]
    if len(filtered) == len(tasks):
        raise KeyError(identifier)
    repository.save(filtered)
delete_task(repository, created.id)
assert repository.load() == []
print("搜索与删除验收通过")